# Model Case Scenario — Investment Simulation (full sweep)

This notebook simulates an investor that trades gold using cached out-of-sample forecasts written by `Modelling Baseline.ipynb` to `Results/{STEM}.predictions.csv`. It runs a **full sweep**: it discovers every prediction sidecar on disk and back-tests **every `(dataset, window scheme, forecast horizon, model)` combination** the baseline produced. For each combination we compute an equity curve over the out-of-sample window and compare it against a buy-and-hold benchmark.

**Strategy specification**

- Starting capital: \$1000
- Long / flat (no shorts)
- Long if `pred > 0`; flat if `pred < 0`; **hold previous position** if `pred` falls inside the 45–55th percentile of the full-sample `y_true` distribution (computed per `dataset × scheme × horizon`)
- Position earns the **realized log-return over the decision interval** (computed from `gold_close`)
- No transaction costs
- **Decision stride = forecast horizon** — non-overlapping holding periods; `stride_bars = horizon_min // BAR_MINUTES` (5 min → 1 bar, 30 min → 6 bars, 60 min → 12 bars)

**Metrics reported per `(dataset × scheme × horizon × model)`**

- Terminal value & total return
- Annualised Sharpe (estimated empirically from the data span)
- Turnover (number of position flips, including the initial entry from cash)
- Buy-and-hold equivalent on the same decision grid

**Caveats** (see Section 7 for details)

- Full-sample percentile thresholds introduce a mild lookahead (defensible, common in the literature).
- Zero transaction costs is an upper bound — appendix sensitivity recommended; the assumption bites hardest at short horizons (high turnover).

## 0. Backtest configuration

Edit the constants below before re-executing. The notebook loads the baseline pipeline (data, features, model factories, walk-forward runner) via `nbformat` **without** running the heavy grid (`CELL 8` of the baseline).

By default the backtest is a **full sweep** — every dataset, window scheme, forecast horizon, and model with a prediction sidecar in `Results/` is simulated. The `DATASETS_FILTER` / `SCHEMES_FILTER` / `MODELS_FILTER` / `HORIZONS` constants are optional narrowing knobs; leave them `None` to back-test everything found on disk.

In [ ]:
# ── Backtest configuration ────────────────────────────────────────────────────
STARTING_CASH        = 1000.0
DEADBAND_PERCENTILES = (0.45, 0.55)   # percentiles of full-sample y_true,
                                      # computed per (dataset, scheme, horizon)

# This notebook runs a FULL SWEEP: it discovers every cached prediction sidecar in
# Results/ and back-tests every (dataset, window scheme, forecast horizon, model)
# combination the baseline produced. The filters below are optional narrowing
# knobs — leave them as None to "simulate everything found on disk".
DATASETS_FILTER = None   # e.g. ["poly", "poly+trad"]; None = every dataset on disk
SCHEMES_FILTER  = None   # e.g. ["fixed300", "expanding"]; None = every scheme on disk
MODELS_FILTER   = None   # e.g. ["pls_ols", "rf_pls"];    None = every model on disk
HORIZONS        = None   # e.g. [5, 30, 60] (minutes);    None = every horizon on disk

BASELINE_NOTEBOOK_PATH = "Modelling Baseline.ipynb"   # relative to this notebook

## 1. Load the baseline pipeline (without running the heavy grid)

We `exec` every code cell of `Modelling Baseline.ipynb` **except** the grid runner (`CELL 8`) and its summary cells (`CELL 9`, `CELL 9.1`). After this cell we have the full data pipeline (`gold`, `X`, `y`, `DATASET_SPECS`), the model factories (`BASE_MODEL_BUILDERS`, `build_model_registry`), and helper utilities such as `artefact_paths`, `compute_metrics`, and `walk_forward` available in the local namespace.

This notebook does **not** call the heavy walk-forward. Section 2 loads cached prediction sidecars written by the baseline.

In [5]:
#check if nbformat is installed; if not, install it via pip and import it;
#otherwise, just import it. This is needed to read and execute the baseline notebook cells.
try:
    import nbformat
except ImportError:
    import subprocess
    import sys
    print("nbformat not found. Installing it via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])

import nbformat as _nbf
import re as _re
from pathlib import Path as _Path

_baseline_path = _Path(BASELINE_NOTEBOOK_PATH)
if not _baseline_path.exists():
    raise FileNotFoundError(
        f"Baseline notebook not found at {_baseline_path.resolve()}.\n"
        "Run this notebook from the same directory as 'Modelling Baseline.ipynb'."
    )

_nb_base = _nbf.read(_baseline_path, as_version=4)
# Match the "# %% ── CELL N : ..." marker for cells we want to skip (heavy grid + summaries).
_skip_re = _re.compile(r"# %% .{0,4} CELL (8|9|9\.1) :")

_executed, _skipped = [], []
for _cell in _nb_base.cells:
    if _cell.cell_type != "code":
        continue
    if _skip_re.search(_cell.source):
        _first_line = _cell.source.splitlines()[0][:80] if _cell.source.splitlines() else "<empty>"
        _skipped.append(_first_line)
        continue
    exec(compile(_cell.source, str(_baseline_path), "exec"), globals())
    _first_line = _cell.source.splitlines()[0][:80] if _cell.source.splitlines() else "<empty>"
    _executed.append(_first_line)

print(f"\nExecuted {len(_executed)} cell(s) from {_baseline_path.name}; "
      f"skipped {len(_skipped)} (grid + summary).")
print("Skipped:")
for _s in _skipped:
    print(f"  - {_s}")


Using Bloomberg workbook: Data\Indicators Data bloomberg.xlsx
Loaded PLS_N_COMPONENTS   : 3
Artefact path             : Feature selection log\pls_n_components_latest.json
Artefact generated_at_utc :  2026-05-14T15:32:50Z
Artefact grid_searched    :  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
✅ Dependencies loaded. TF: True
✅ Traditional predictor helpers loaded.
FE_INPUT_MODE  : raw_panel
Latest panel   : polymarket_panel_filtered_2026-05-07.csv  (data date = 2026-05-07)
X_raw shape    : (7171, 690)   range: 2026-04-01 00:00:00 → 2026-05-07 22:00:00
Latest Bloomberg stationary panel : bloomberg_panel_stationary_2026-05-07.csv  (data date = 2026-05-07)
Latest Bloomberg metadata        : bloomberg_stationarity_metadata_2026-05-07.json
Bloomberg stationarity: 9 differenced (9 logret, 0 diff), 0 kept in levels  | train_fraction=1.0 | max_ffill_bars=None
openpyxl is already installed.
Bloomberg sheets loaded: ['GOLD USD SPOT PER OZ', 'CRUDE OIL (WTI) FUTURES PRI

## 2. Discover & load cached predictions (full sweep)

This section scans `Results/` for `*.predictions.csv` sidecars written by `Modelling Baseline.ipynb` and reads each run's identity — model, window scheme, dataset, forecast horizon — **straight from the file name**. Because the model set is derived from the files actually on disk, the loaded set always matches what the baseline produced; there is no hardcoded model list that can drift out of sync.

Every discovered `(dataset, scheme, horizon, model)` combination is loaded unless excluded by the optional filters. When two sidecars exist for the same combination the freshest one (matching the current `data_sha1`, else newest by date) is used; stale sidecars are flagged with a warning but still loaded. The notebook never trains.

**Operational note** — the baseline grid runs at a single `RETURN_HORIZON_MIN`. To populate multiple horizons, run `Modelling Baseline.ipynb` once per horizon (set `RETURN_HORIZON_MIN` to each value) against the same data panel; each run appends sidecars for every `(dataset, scheme, model)` at that horizon.

In [ ]:
# ── Discover & load cached walk-forward predictions (full sweep, no training) ──
import re
import pandas as pd
from collections import defaultdict

# --- regex for baseline artefact stems ----------------------------------------
# Stem written by Modelling Baseline.ipynb (CELL 7, artefact_paths):
#   {MODEL_UPPER}_{scheme}_{dataset_tag}_h{H}m_f{n_features}_{data_date}
# scheme and dataset are closed sets registered by the baseline; sort longest-first
# so 'poly+trad' / 'poly_only' match before 'poly'. Model is non-greedy and may
# contain underscores (e.g. PLS_OLS, LASSO_CV) — the closed-set scheme/dataset
# anchors plus backtracking resolve it unambiguously.
_scheme_alt  = "|".join(re.escape(s) for s in sorted(WINDOW_SCHEMES, key=len, reverse=True))
_dataset_alt = "|".join(re.escape(d) for d in sorted(DATASET_SPECS, key=len, reverse=True))
_stem_re = re.compile(
    rf"^(?P<model>.+?)_(?P<scheme>{_scheme_alt})_(?P<dataset>{_dataset_alt})"
    rf"_h(?P<h>\d+)m_f(?P<f>\d+)_(?P<date>\d{{4}}-\d{{2}}-\d{{2}})$"
)

# --- discover every prediction sidecar in Results/ ----------------------------
_parsed = []
for _p in sorted(RESULTS_DIR.glob("*.predictions.csv")):
    _stem = _p.name[:-len(".predictions.csv")]
    _m = _stem_re.match(_stem)
    if _m:
        _parsed.append({
            "path": _p, "stem": _stem, "model": _m["model"].lower(),
            "scheme": _m["scheme"], "dataset": _m["dataset"],
            "horizon": int(_m["h"]), "n_features": int(_m["f"]), "date": _m["date"],
        })
    else:
        print(f"NOTE  sidecar name does not match the baseline stem pattern, ignored: {_p.name}")

if not _parsed:
    raise RuntimeError(
        f"No parseable '*.predictions.csv' sidecars found in {RESULTS_DIR}/.\n"
        "Run 'Modelling Baseline.ipynb' end-to-end first (once per horizon — set "
        "RETURN_HORIZON_MIN to each value you want) so it writes prediction sidecars."
    )

# --- every discovered horizon must be a whole number of bars ------------------
for _h in sorted({d["horizon"] for d in _parsed}):
    if _h % BAR_MINUTES != 0:
        raise ValueError(f"Discovered horizon {_h} min is not a multiple of "
                         f"BAR_MINUTES={BAR_MINUTES}; cannot derive a decision stride.")

# --- apply optional filters (None = keep everything discovered) ---------------
def _keep(d):
    return ((DATASETS_FILTER is None or d["dataset"] in set(DATASETS_FILTER))
            and (SCHEMES_FILTER is None or d["scheme"]  in set(SCHEMES_FILTER))
            and (MODELS_FILTER  is None or d["model"]   in set(MODELS_FILTER))
            and (HORIZONS       is None or d["horizon"] in set(HORIZONS)))

_matches = [d for d in _parsed if _keep(d)]

print(f"Sidecars in {RESULTS_DIR}/ : {len(_parsed)} parsed, {len(_matches)} kept after filters")
print(f"  datasets discovered : {sorted({d['dataset'] for d in _parsed})}")
print(f"  schemes  discovered : {sorted({d['scheme']  for d in _parsed})}")
print(f"  horizons discovered : {sorted({d['horizon'] for d in _parsed})}")
print(f"  models   discovered : {sorted({d['model']   for d in _parsed})}\n")


def _meta_data_sha1(stem):
    """data_sha1 recorded in the run's metadata JSON, or None if unavailable."""
    _mp = RESULTS_DIR / f"{stem}.json"
    if not _mp.exists():
        return None
    try:
        return json.loads(_mp.read_text()).get("data_sha1")
    except Exception:
        return None


# --- group by (dataset, scheme, horizon, model); pick the best sidecar each ---
_groups = defaultdict(list)
for d in _matches:
    _groups[(d["dataset"], d["scheme"], d["horizon"], d["model"])].append(d)

predictions = {}   # (dataset, scheme, horizon, model) -> {"y_true","y_pred","timestamps"}
_notes = []

for _key, _cand in sorted(_groups.items()):
    _ds, _sc, _h, _model = _key
    _fresh = [d for d in _cand if _meta_data_sha1(d["stem"]) == DATA_HASH]
    _pool  = _fresh if _fresh else _cand
    _chosen = max(_pool, key=lambda d: d["date"])          # newest by data_date
    if not _fresh:
        _notes.append(f"WARN  {_ds}/{_sc}/h{_h}m/{_model}: no sidecar matches the current "
                       f"data panel (DATA_HASH); loaded newest ({_chosen['date']}) -- may be stale")
    elif len(_cand) > 1:
        _notes.append(f"INFO  {_ds}/{_sc}/h{_h}m/{_model}: {len(_cand)} sidecars present; "
                       f"used {_chosen['stem']}")

    _df = pd.read_csv(_chosen["path"], parse_dates=["timestamp"])
    predictions[_key] = {
        "y_true":     _df["y_true"].to_numpy(),
        "y_pred":     _df["y_pred"].to_numpy(),
        "timestamps": list(_df["timestamp"]),
    }
    _mt = compute_metrics(predictions[_key]["y_true"], predictions[_key]["y_pred"])
    print(f"OK   {_ds:14s} / {_sc:9s} / h{_h:>3}m / {_model:9s} : "
          f"{len(_df):>7,} obs  rmse={_mt['rmse']:.2e}  dir_acc={_mt['dir_acc']:.3f}")

# --- per-dataset rollup + notes -----------------------------------------------
print()
_by_dataset = defaultdict(int)
for (_ds, _sc, _h, _model) in predictions:
    _by_dataset[_ds] += 1
for _ds in sorted(_by_dataset):
    print(f"dataset {_ds:14s} : {_by_dataset[_ds]} (scheme, horizon, model) combination(s)")

if _notes:
    print()
    for _n in _notes:
        print(_n)

if not predictions:
    _combos = sorted({(d["dataset"], d["scheme"], d["horizon"], d["model"]) for d in _parsed})
    raise RuntimeError(
        "No cached prediction sidecars matched the configured filters "
        f"(DATASETS_FILTER={DATASETS_FILTER}, SCHEMES_FILTER={SCHEMES_FILTER}, "
        f"MODELS_FILTER={MODELS_FILTER}, HORIZONS={HORIZONS}).\n"
        "Sidecars present in Results/ cover these (dataset, scheme, horizon, model):\n  "
        + "\n  ".join(f"{ds} / {sc} / h{hz}m / {ml}" for ds, sc, hz, ml in _combos)
        + "\nSet the *_FILTER / HORIZONS constants to match what is on disk (or to None), "
          "or run the baseline for the missing combinations."
    )

print(f"\nTotal loaded: {len(predictions)} (dataset, scheme, horizon, model) prediction set(s).")

## 3. Backtest simulator

For a given stride **S** (in 5-min bars):

1. **Decision timestamps** = prediction timestamps sub-sampled at stride S (so successive holding periods do not overlap).
2. **Realized log-return** for stride k is `log(gold_close[t_{k+1}]) − log(gold_close[t_k])` where `t_{k+1} = t_k + S` bars.
3. **Signal** at decision k:
   - if `pred_k ∈ [P_lo, P_hi]` → hold previous position (deadband; default is *cash* before the first non-deadband signal)
   - elif `pred_k > 0` → long
   - else → flat
4. **Position** earns the realized log-return for the next stride.
5. **Equity** is `STARTING_CASH × exp(cumsum(position × realized_logret))`.

The stride equals the forecast horizon (`stride_bars = horizon // BAR_MINUTES`). The deadband thresholds `[P_lo, P_hi]` are the 45–55th percentiles of the full-sample `y_true` of each `(dataset, scheme, horizon)` group; see Section 7 for the lookahead caveat.

In [ ]:
import numpy as np
import pandas as pd


def _full_sample_deadband(y_true_all, p_low, p_high):
    return float(np.quantile(y_true_all, p_low)), float(np.quantile(y_true_all, p_high))


def _annualisation_factor(decision_times):
    """Decision samples per year, inferred from the actual sampling span."""
    if len(decision_times) < 2:
        return 1.0
    span_seconds = (decision_times[-1] - decision_times[0]).total_seconds()
    if span_seconds <= 0:
        return 1.0
    seconds_per_year = 365.25 * 24 * 3600
    return len(decision_times) * seconds_per_year / span_seconds


def simulate_strategy(
    timestamps,
    y_pred,
    gold_close,
    stride,
    deadband_lo,
    deadband_hi,
    starting_cash=1000.0,
):
    """Long/flat backtest at a fixed non-overlapping stride.

    Parameters
    ----------
    timestamps : sequence of pd.Timestamp aligned with y_pred
    y_pred     : 1-D array of forecasts (forecast horizon varies; caller passes the matching stride)
    gold_close : pd.Series of gold close prices indexed by datetime
    stride     : positive integer, decision interval in 5-min bars
    deadband_lo, deadband_hi : float thresholds; if deadband_lo <= pred <= deadband_hi -> hold
    starting_cash : initial equity in same units used downstream
    """
    timestamps = pd.DatetimeIndex(timestamps)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(timestamps)

    decision_idx = np.arange(0, n - stride, stride, dtype=int)
    if len(decision_idx) == 0:
        raise ValueError(f"Not enough timestamps ({n}) for stride={stride}.")
    decision_times = timestamps[decision_idx]
    next_times     = timestamps[decision_idx + stride]
    decision_preds = y_pred[decision_idx]

    closes      = gold_close.reindex(decision_times).to_numpy(dtype=float)
    closes_next = gold_close.reindex(next_times).to_numpy(dtype=float)

    # If any close is missing (rare — usually only at panel edges), drop those steps.
    _valid = ~(np.isnan(closes) | np.isnan(closes_next))
    if not _valid.all():
        decision_times = decision_times[_valid]
        next_times     = next_times[_valid]
        decision_preds = decision_preds[_valid]
        closes         = closes[_valid]
        closes_next    = closes_next[_valid]

    realized_logrets = np.log(closes_next) - np.log(closes)

    positions = np.zeros(len(decision_preds), dtype=int)
    prev = 0
    for i, pred in enumerate(decision_preds):
        if deadband_lo <= pred <= deadband_hi:
            positions[i] = prev          # hold previous
        elif pred > 0:
            positions[i] = 1             # long
        else:
            positions[i] = 0             # flat
        prev = positions[i]

    strategy_logrets = positions * realized_logrets
    equity_strategy  = starting_cash * np.exp(np.cumsum(strategy_logrets))
    equity_bh        = starting_cash * np.exp(np.cumsum(realized_logrets))

    # Trade count = number of position transitions, counting the initial entry from cash.
    n_trades = int((np.diff(np.concatenate(([0], positions))) != 0).sum())

    ann = _annualisation_factor(decision_times)
    sharpe_strategy = (
        float(strategy_logrets.mean() / strategy_logrets.std() * np.sqrt(ann))
        if strategy_logrets.std() > 0 else 0.0
    )
    sharpe_bh = (
        float(realized_logrets.mean() / realized_logrets.std() * np.sqrt(ann))
        if realized_logrets.std() > 0 else 0.0
    )

    pct_time_long = float(positions.mean())

    return {
        "decision_times":   decision_times,
        "positions":        positions,
        "realized_logrets": realized_logrets,
        "strategy_logrets": strategy_logrets,
        "equity_strategy":  equity_strategy,
        "equity_bh":        equity_bh,
        "terminal_value":   float(equity_strategy[-1]),
        "total_return":     float(equity_strategy[-1] / starting_cash - 1),
        "bh_terminal":      float(equity_bh[-1]),
        "bh_total_return":  float(equity_bh[-1] / starting_cash - 1),
        "n_trades":         n_trades,
        "ann_factor":       ann,
        "sharpe":           sharpe_strategy,
        "bh_sharpe":        sharpe_bh,
        "pct_time_long":    pct_time_long,
    }


print("✅ Simulator loaded.")


## 4. Run simulations: full sweep

For every loaded `(dataset, scheme, horizon, model)` combination we run `simulate_strategy` with `stride = horizon // BAR_MINUTES` and store the result in `backtest_results[(dataset, scheme, horizon, model)]`. The deadband band is computed once per `(dataset, scheme, horizon)` group. Any combination whose simulation raises is reported and skipped, so one bad combo never aborts the sweep. The combined `summary_df` feeds the table in Section 5.

In [ ]:
# `gold` is built by the baseline (CELL 3); gold_close is the spot close
# (horizon-independent), shared by every simulation.
_gold_close = gold["gold_close"].sort_index()
_gold_close = _gold_close[~_gold_close.index.duplicated(keep="last")]

# Deadband band per (dataset, scheme, horizon): y_true is the h-min-ahead return and
# its OOS coverage depends on the walk-forward scheme, so the percentile band is
# group-specific. Models within a group share the same y_true.
_deadband = {}
for (_ds, _sc, _h, _model), _res in predictions.items():
    _gkey = (_ds, _sc, _h)
    if _gkey not in _deadband:
        _deadband[_gkey] = _full_sample_deadband(_res["y_true"], *DEADBAND_PERCENTILES)

backtest_results = {}    # (dataset, scheme, horizon, model) -> sim dict
summary_rows = []
_sim_errors = []

for _key in sorted(predictions):
    _ds, _sc, _h, _model = _key
    _res = predictions[_key]
    _stride = max(1, _h // BAR_MINUTES)                    # decision stride = forecast horizon
    _lo, _hi = _deadband[(_ds, _sc, _h)]
    try:
        _sim = simulate_strategy(
            timestamps=_res["timestamps"], y_pred=_res["y_pred"],
            gold_close=_gold_close, stride=_stride,
            deadband_lo=_lo, deadband_hi=_hi, starting_cash=STARTING_CASH,
        )
    except Exception as _e:
        _sim_errors.append(f"SKIP  {_ds}/{_sc}/h{_h}m/{_model}: {type(_e).__name__}: {_e}")
        continue
    backtest_results[_key] = _sim
    summary_rows.append({
        "dataset":         _ds,
        "scheme":          _sc,
        "horizon_min":     _h,
        "stride_bars":     _stride,
        "model":           _model,
        "n_decisions":     len(_sim["decision_times"]),
        "n_trades":        _sim["n_trades"],
        "pct_time_long":   _sim["pct_time_long"],
        "terminal":        _sim["terminal_value"],
        "total_return":    _sim["total_return"],
        "sharpe":          _sim["sharpe"],
        "bh_terminal":     _sim["bh_terminal"],
        "bh_total_return": _sim["bh_total_return"],
        "bh_sharpe":       _sim["bh_sharpe"],
    })

if _sim_errors:
    print("Simulation skips:")
    for _e in _sim_errors:
        print(" ", _e)
    print()

if not summary_rows:
    raise RuntimeError("No simulations produced results — see the SKIP messages above.")

summary_df = (
    pd.DataFrame(summary_rows)
      .sort_values(["dataset", "horizon_min", "scheme", "model"])
      .reset_index(drop=True)
)
print(f"Backtest complete: {len(summary_df)} (dataset, scheme, horizon, model) combination(s).\n")
with pd.option_context("display.float_format", "{:,.4f}".format,
                       "display.width", 200, "display.max_rows", None):
    print(summary_df.to_string(index=False))

## 5. Summary table

The complete per-combination metrics table — one row per `(dataset, scheme, horizon, model)` — is saved to `Results/backtest_summary_full_sweep.csv` for the thesis appendix.

In [ ]:
_out_summary = Path("Results") / "backtest_summary_full_sweep.csv"
_out_summary.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(_out_summary, index=False)
print(f"Saved {len(summary_df)}-row sweep summary to {_out_summary}")
summary_df

## 6. Equity-curve plots — one figure per dataset

For each dataset, one figure with **one subplot per forecast horizon**. Within a subplot every `(scheme, model)` equity curve is overlaid — **colour encodes the model, line style encodes the window scheme** — alongside a faint buy-and-hold benchmark per scheme. Each figure is saved as `Results/backtest_equity_curves_{dataset}.png`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

_keys = sorted(backtest_results)
_datasets = sorted({k[0] for k in _keys})

# Stable encoding shared across every figure: colour = model, line style = scheme.
_all_models  = sorted({k[3] for k in _keys})
_all_schemes = sorted({k[1] for k in _keys})
_palette = plt.get_cmap("tab10").colors
_model_color  = {m: _palette[i % len(_palette)] for i, m in enumerate(_all_models)}
_style_cycle  = ["-", "--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 1))]
_scheme_style = {s: _style_cycle[i % len(_style_cycle)] for i, s in enumerate(_all_schemes)}

for _ds in _datasets:
    _ds_keys = [k for k in _keys if k[0] == _ds]
    _hs = sorted({k[2] for k in _ds_keys})
    fig, axes = plt.subplots(len(_hs), 1, figsize=(12, 3.8 * len(_hs)), squeeze=False)
    axes = axes[:, 0]

    for ax, _h in zip(axes, _hs):
        _stride = max(1, _h // BAR_MINUTES)
        _bh_seen = set()
        for _key in sorted(k for k in _ds_keys if k[2] == _h):
            _sc, _model = _key[1], _key[3]
            _sim = backtest_results[_key]
            ax.plot(_sim["decision_times"], _sim["equity_strategy"],
                    color=_model_color[_model], linestyle=_scheme_style[_sc],
                    linewidth=1.1, alpha=0.9)
            if _sc not in _bh_seen:                         # one buy & hold per scheme
                ax.plot(_sim["decision_times"], _sim["equity_bh"],
                        color="black", linestyle=_scheme_style[_sc],
                        linewidth=1.0, alpha=0.35)
                _bh_seen.add(_sc)
        ax.axhline(STARTING_CASH, color="grey", linewidth=0.7, alpha=0.5)
        ax.set_title(f"horizon {_h} min   (decision stride {_stride} bar(s))", fontsize=10)
        ax.set_ylabel("Equity ($)")
        ax.grid(alpha=0.3)
    axes[-1].set_xlabel("Date")

    # Two compact legends on the top subplot: colour -> model, line style -> scheme.
    _model_handles = [Line2D([0], [0], color=_model_color[m], lw=1.8, label=m)
                      for m in _all_models]
    _scheme_handles = [Line2D([0], [0], color="black", lw=1.5,
                               linestyle=_scheme_style[s], label=s)
                       for s in _all_schemes]
    _scheme_handles.append(Line2D([0], [0], color="black", lw=1.0, alpha=0.35,
                                   linestyle="-", label="buy & hold (per scheme)"))
    _leg_models = axes[0].legend(handles=_model_handles, title="model (colour)",
                                 loc="upper left", fontsize=8, ncol=2)
    axes[0].add_artist(_leg_models)
    axes[0].legend(handles=_scheme_handles, title="scheme (line style)",
                   loc="upper right", fontsize=8)

    fig.suptitle(f"Equity curves — dataset = {_ds}   "
                 f"(start ${STARTING_CASH:,.0f}, deadband "
                 f"{DEADBAND_PERCENTILES[0]:.0%}–{DEADBAND_PERCENTILES[1]:.0%})",
                 fontsize=12, y=1.002)
    fig.tight_layout()
    _out_fig = Path("Results") / f"backtest_equity_curves_{_ds}.png"
    fig.savefig(_out_fig, dpi=140, bbox_inches="tight")
    print(f"Saved {_out_fig}   ({len(_hs)} horizon facet(s))")
    plt.show()

## 7. Notes & caveats

- **No transaction costs.** Strategy returns are gross. Even a flat 1 bp per position flip can flip the sign of a high-turnover strategy's Sharpe. Inspect `n_trades` in the summary table to gauge sensitivity; consider an appendix run with a small cost (e.g. multiply the equity update by `(1 − cost)` on each flip).
- **Stride = forecast horizon.** Each strategy decides once per forecast horizon and holds for exactly that horizon (non-overlapping). Short horizons therefore decide very frequently — a 5-min horizon means a decision every bar — producing high turnover, so the zero-transaction-cost assumption bites hardest there. Read short-horizon results with that in mind.
- **Full-sample deadband percentile.** The 45–55th percentile of `y_true` is computed on the entire OOS sample (per `dataset × scheme × horizon`), which uses knowledge that would not have been available in real time. The bias is small in practice — it only re-classifies *which* trades are no-ops, not the sign of any executed trade — but a robust appendix run can recompute the percentile from a rolling training-window estimate.
- **Sharpe annualisation.** Estimated empirically as `samples_per_year = n_decisions × (year / data_span)` and Sharpe is `mean / std × √samples_per_year`. This is robust to weekend / overnight gaps and irregular sampling but assumes the OOS window is representative of a "year" of returns.
- **Cached prediction dependency.** This notebook loads OOS predictions from `Results/{STEM}.predictions.csv` sidecars written by `Modelling Baseline.ipynb`. Run the baseline end-to-end first (once per horizon); combinations with missing sidecars are simply absent from the sweep, and stale sidecars (data-hash mismatch) are flagged with a warning.